# 01 — Data Understanding

## Objective

This notebook performs an initial assessment of the raw **Diabetes 130-US Hospitals for Years 1999-2008** dataset.

The goal is to understand:

- Dataset structure and dimensions
- Feature types and cardinality
- Missing and unknown values
- Target distribution
- Duplicate records
- Constant and near-constant features
- Basic numerical statistics
- Potential data quality issues

No data cleaning, feature engineering, feature selection, or modeling is performed in this phase.

## cell 2 - Imports & Configuration ##


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

## 1. Load Raw Dataset

In [3]:
DATA_PATH = Path("../data/raw/diabetic_data.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (101766, 50)


In [4]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [6]:
df.dtypes.value_counts()

object    37
int64     13
Name: count, dtype: int64

## 3. Column Overview

We inspect the column names, data types, and number of unique values to identify categorical variables, identifiers, and high-cardinality features.

In [7]:
column_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(dropna=False),
    "n_missing": df.isna().sum()
})

column_summary

,dtype,n_unique,n_missing
encounter_id,int64,101766,0
patient_nbr,int64,71518,0
race,object,6,0
gender,object,3,0
age,object,10,0
weight,object,10,0
admission_type_id,int64,8,0
discharge_disposition_id,int64,26,0
admission_source_id,int64,17,0
time_in_hospital,int64,14,0


## Unknown and Missing Values

The raw dataset uses `?` as an unknown-value marker in addition to standard missing values (`NaN`).

In [8]:
unknown_summary = pd.DataFrame({
    "question_mark": (df == "?").sum(),
    "NaN": df.isna().sum()
})

unknown_summary = unknown_summary[
    (unknown_summary["question_mark"] > 0) |
    (unknown_summary["NaN"] > 0)
].sort_values("question_mark", ascending=False)

unknown_summary

,question_mark,NaN
weight,98569,0
medical_specialty,49949,0
payer_code,40256,0
race,2273,0
diag_3,1423,0
diag_2,358,0
diag_1,21,0
max_glu_serum,0,96420
A1Cresult,0,84748


## Target Distribution

The original target contains three categories:

- `NO`: No readmission
- `<30`: Readmitted within 30 days
- `>30`: Readmitted after 30 days

For the final modeling task, `<30` and `>30` will later be combined into the positive class.

In [9]:
df["readmitted"].value_counts(dropna=False)

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [10]:
df["readmitted"].value_counts(normalize=True).mul(100).round(2)

readmitted
NO     53.91
>30    34.93
<30    11.16
Name: proportion, dtype: float64

## Duplicate Records

We check for exact duplicate rows in the raw dataset before making any cleaning decisions.

In [11]:
duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")

Exact duplicate rows: 0


## Constant and Near-Constant Features

Constant features contain a single unique value and provide no predictive information.

Near-constant features are also flagged for further investigation during the cleaning phase.

In [12]:
constant_features = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

near_constant_features = []

for col in df.columns:
    top_frequency = df[col].value_counts(dropna=False, normalize=True).iloc[0]
    if top_frequency >= 0.995 and col not in constant_features:
        near_constant_features.append(col)

print("Constant features:")
print(constant_features)

print("\nNear-constant features (>= 99.5% same value):")
print(near_constant_features)

Constant features:
['examide', 'citoglipton']

Near-constant features (>= 99.5% same value):
['chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


## Numerical Feature Statistics

We inspect the distribution of numerical variables to identify unusual ranges and potential outliers.

No outliers are removed at this stage.

In [13]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,1.652016e+08,1.026403e+08,12522.0,84961194.0,152388987.0,2.302709e+08,443867222.0
patient_nbr,101766.0,5.433040e+07,3.869636e+07,135.0,23413221.0,45505143.0,8.754595e+07,189502619.0
admission_type_id,101766.0,2.024006e+00,1.445403e+00,1.0,1.0,1.0,3.000000e+00,8.0
discharge_disposition_id,101766.0,3.715642e+00,5.280166e+00,1.0,1.0,1.0,4.000000e+00,28.0
admission_source_id,101766.0,5.754437e+00,4.064081e+00,1.0,1.0,7.0,7.000000e+00,25.0
time_in_hospital,101766.0,4.395987e+00,2.985108e+00,1.0,2.0,4.0,6.000000e+00,14.0
num_lab_procedures,101766.0,4.309564e+01,1.967436e+01,1.0,31.0,44.0,5.700000e+01,132.0
num_procedures,101766.0,1.339730e+00,1.705807e+00,0.0,0.0,1.0,2.000000e+00,6.0
num_medications,101766.0,1.602184e+01,8.127566e+00,1.0,10.0,15.0,2.000000e+01,81.0
number_outpatient,101766.0,3.693572e-01,1.267265e+00,0.0,0.0,0.0,0.000000e+00,42.0


## High-Cardinality Categorical Features

Categorical features with many unique values require special attention during preprocessing and may require domain-specific handling.

In [14]:
categorical_cols = df.select_dtypes(include="object").columns

cardinality = (
    df[categorical_cols]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .to_frame("n_unique")
)

cardinality

,n_unique
diag_3,790
diag_2,749
diag_1,717
medical_specialty,73
payer_code,18
age,10
weight,10
race,6
glipizide,4
glyburide-metformin,4


## Data Understanding Summary

The raw dataset has now been inspected for:

- Structure and data types
- Missing and unknown values
- Target distribution
- Duplicate records
- Constant and near-constant features
- Numerical statistics
- Categorical cardinality

No cleaning or transformation has been performed in this phase.

In [15]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Exact duplicates: {df.duplicated().sum():,}")
print(f"Object columns: {df.select_dtypes(include='object').shape[1]}")
print(f"Numeric columns: {df.select_dtypes(include=np.number).shape[1]}")

Rows: 101,766
Columns: 50
Exact duplicates: 0
Object columns: 37
Numeric columns: 13


## Conclusion

The raw dataset contains 101,766 patient encounters and 50 features.

The initial assessment identified:

- 37 categorical and 13 numerical columns
- Unknown values represented by `?`
- No exact duplicate rows
- Two constant features: `examide` and `citoglipton`
- Several near-constant medication features
- High-cardinality diagnosis and medical-specialty variables
- A three-class readmission target with moderate class imbalance

These findings will guide the data-cleaning decisions in the next phase.

No data transformation or feature removal has been performed during this phase.